In [7]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


# Pré Processamento

In [8]:
from pathlib import Path

from causal_discovery import CausalPreprocessor, load_daily_delhi_climate, summarize_ensemble
from causal_discovery.methods import (
    run_classical_granger,
    run_dynotears,
    run_heterogeneous_causal_discovery,
    run_lpcmci,
    run_neural_granger,
    run_pcmci,
    run_score_based_search,
    run_var_lingam,
    )

DATA_PATH = Path("DailyDelhiClimateTrain.csv")
SELECTED_COLUMNS = ["meantemp", "humidity", "wind_speed", "meanpressure"]

raw_data = load_daily_delhi_climate(DATA_PATH)
data = raw_data[SELECTED_COLUMNS].copy()

preprocessor = CausalPreprocessor(
    data,
    significance_level=0.05,
    decomposition_period=30,
    )
processed_data = preprocessor.fit_transform(
    make_stationary=True,
    normalize=True,
    remove_trend=False,
    max_diffs=2,
    )

max_lag = 5
preprocessing_summary = preprocessor.summary()

print("--- Dados Originais ---")
print(data.head())
print("\n--- Resumo do Pré-processamento ---")
print(preprocessing_summary)
print("\n--- Dados Processados ---")
print(processed_data.head())

--- Dados Originais ---
             meantemp   humidity  wind_speed  meanpressure
date                                                      
2013-01-01  10.000000  84.500000    0.000000   1015.666667
2013-01-02   7.400000  92.000000    2.980000   1017.800000
2013-01-03   7.166667  87.000000    4.633333   1018.666667
2013-01-04   8.666667  71.333333    1.233333   1017.166667
2013-01-05   6.000000  86.833333    3.700000   1016.500000

--- Resumo do Pré-processamento ---
         column  differencing_order
0      meantemp                   1
1      humidity                   0
2    wind_speed                   0
3  meanpressure                   0

--- Dados Processados ---
            meantemp  humidity  wind_speed  meanpressure
date                                                    
2013-01-02 -1.556049  1.864438   -0.839570      0.037166
2013-01-03 -0.139645  1.566076   -0.476847      0.041975
2013-01-04  0.897720  0.631208   -1.222768      0.033652
2013-01-05 -1.595947  1.556131   -

# Instalações

In [9]:
print("Dependências centralizadas em requirements.txt e instaladas na célula 1.")

Dependências centralizadas em requirements.txt e instaladas na célula 1.


# PCMCI

In [10]:
pcmci_results = run_pcmci(
    processed_data,
    max_lag=max_lag,
    pc_alpha=0.05,
    alpha_level=0.05,
    )

lpcmci_results = run_lpcmci(
    processed_data,
    max_lag=max_lag,
    pc_alpha=0.05,
    )

print("--- Resultados PCMCI ---")
print(pcmci_results.head())
print(f"Total de links PCMCI: {len(pcmci_results)}")

print("\n--- Resultados LPCMCI ---")
print(lpcmci_results.head())
print(f"Total de links LPCMCI: {len(lpcmci_results)}")

--- Resultados PCMCI ---
     source    target  lag     score        p_value method  q_value
0  meantemp  meantemp    1 -0.117261   7.801202e-06  PCMCI      NaN
1  meantemp  meantemp    2 -0.121398   3.694770e-06  PCMCI      NaN
2  meantemp  meantemp    3 -0.165039   2.801788e-10  PCMCI      NaN
3  meantemp  meantemp    4 -0.088027   8.155445e-04  PCMCI      NaN
4  humidity  meantemp    0 -0.644410  1.809343e-170  PCMCI      NaN
Total de links PCMCI: 21

--- Resultados LPCMCI ---
     source    target  lag     score        p_value  method  q_value
0  meantemp  meantemp    1 -0.116584   8.619565e-06  LPCMCI      NaN
1  meantemp  meantemp    2 -0.066265   1.160660e-02  LPCMCI      NaN
2  meantemp  meantemp    3 -0.102062   9.947387e-05  LPCMCI      NaN
3  humidity  meantemp    0 -0.644989  3.204710e-171  LPCMCI      NaN
4  meantemp  humidity    0 -0.644989  3.204710e-171  LPCMCI      NaN
Total de links LPCMCI: 17


# Baselines supervisionados

In [11]:
classical_granger_results = run_classical_granger(
    processed_data,
    max_lag=max_lag,
    alpha=0.05,
    )

neural_granger_results = run_neural_granger(
    processed_data,
    max_lag=max_lag,
    max_iter=100,
    perm_repeats=5,
    score_threshold=0.0,
    )

print("--- Resultados Classical Granger ---")
print(classical_granger_results.head())
print(f"Total de links Classical Granger: {len(classical_granger_results)}")

print("\n--- Resultados Neural Granger ---")
print(neural_granger_results.head())
print(f"Total de links Neural Granger: {len(neural_granger_results)}")

--- Resultados Classical Granger ---
     source    target  lag      score       p_value            method
0  meantemp  humidity    1  24.560057  8.045153e-07  ClassicalGranger
1  meantemp  humidity    2  12.497671  4.153893e-06  ClassicalGranger
2  meantemp  humidity    3  13.547007  1.016709e-08  ClassicalGranger
3  meantemp  humidity    4   7.635041  4.366593e-06  ClassicalGranger
4  meantemp  humidity    5   4.952223  1.681471e-04  ClassicalGranger
Total de links Classical Granger: 25

--- Resultados Neural Granger ---
         source    target  lag     score  p_value            method
0      meantemp  meantemp    1  0.062553      NaN  NeuralGrangerMLP
1      humidity  meantemp    1  0.033206      NaN  NeuralGrangerMLP
2    wind_speed  meantemp    1  0.020400      NaN  NeuralGrangerMLP
3  meanpressure  meantemp    1  0.001025      NaN  NeuralGrangerMLP
4      meantemp  meantemp    2  0.030570      NaN  NeuralGrangerMLP
Total de links Neural Granger: 78


# LiNGAM, score-based e séries heterogêneas

In [12]:
var_lingam_results = run_var_lingam(
    processed_data,
    max_lag=max_lag,
    )

dynotears_results = run_dynotears(
    processed_data,
    max_lag=max_lag,
    lambda1=0.01,
    max_iter=50,
    w_threshold=0.05,
    standardize=True,
    )

score_based_results = run_score_based_search(
    processed_data,
    max_lag=max_lag,
    min_bic_improvement=1.0,
    )

heterogeneous_results = run_heterogeneous_causal_discovery(
    processed_data,
    max_lag=max_lag,
    alpha=0.05,
    n_segments=4,
    min_segment_votes=2,
    )

print("--- Resultados VAR-LiNGAM ---")
print(var_lingam_results.head())
print(f"Total de links VAR-LiNGAM: {len(var_lingam_results)}")

print("\n--- Resultados DYNOTEARS ---")
print(dynotears_results.head())
print(f"Total de links DYNOTEARS: {len(dynotears_results)}")

print("\n--- Resultados Score-Based (BIC) ---")
print(score_based_results.head())
print(f"Total de links Score-Based: {len(score_based_results)}")

print("\n--- Resultados Heterogeneous Temporal Discovery ---")
print(heterogeneous_results.head())
print(f"Total de links Heterogeneous: {len(heterogeneous_results)}")

--- Resultados VAR-LiNGAM ---
     source      target  lag     score  p_value     method
0  humidity    meantemp    0 -1.338035      NaN  VARLiNGAM
1  meantemp  wind_speed    0 -0.176527      NaN  VARLiNGAM
2  humidity  wind_speed    0 -0.688996      NaN  VARLiNGAM
3  meantemp    meantemp    1 -0.057207      NaN  VARLiNGAM
4  humidity    meantemp    1  1.293498      NaN  VARLiNGAM
Total de links VAR-LiNGAM: 9

--- Resultados DYNOTEARS ---
       source      target  lag     score  p_value     method
0    humidity    meantemp    0 -1.313523      NaN  DYNOTEARS
1    humidity  wind_speed    0 -0.393161      NaN  DYNOTEARS
2  wind_speed    meantemp    0 -0.105335      NaN  DYNOTEARS
3    meantemp    meantemp    1 -0.094696      NaN  DYNOTEARS
4    humidity    meantemp    1  1.162167      NaN  DYNOTEARS
Total de links DYNOTEARS: 17

--- Resultados Score-Based (BIC) ---
     source    target  lag     score        p_value         method  \
0  meantemp  meantemp    1 -0.205642   9.358052e-15  S

# Ensemble

In [13]:
all_results = [
    pcmci_results,
    lpcmci_results,
    classical_granger_results,
    neural_granger_results,
    var_lingam_results,
    dynotears_results,
    score_based_results,
    heterogeneous_results,
    ]

method_sizes = {
    "PCMCI": len(pcmci_results),
    "LPCMCI": len(lpcmci_results),
    "ClassicalGranger": len(classical_granger_results),
    "NeuralGrangerMLP": len(neural_granger_results),
    "VARLiNGAM": len(var_lingam_results),
    "DYNOTEARS": len(dynotears_results),
    "ScoreBasedBIC": len(score_based_results),
    "HeterogeneousTemporalGranger": len(heterogeneous_results),
    }

ensemble_summary = summarize_ensemble(all_results, min_votes=2)

print("--- Quantidade de links por método ---")
print(method_sizes)
print("\n--- Sumário do Ensemble (mínimo de 2 votos) ---")
print(ensemble_summary.head(20))

--- Quantidade de links por método ---
{'PCMCI': 21, 'LPCMCI': 17, 'ClassicalGranger': 25, 'NeuralGrangerMLP': 78, 'VARLiNGAM': 9, 'DYNOTEARS': 17, 'ScoreBasedBIC': 11, 'HeterogeneousTemporalGranger': 14}

--- Sumário do Ensemble (mínimo de 2 votos) ---
        source        target  lag  \
0     humidity    wind_speed    1   
1     meantemp      humidity    1   
2     humidity    wind_speed    2   
3     humidity      meantemp    1   
4     meantemp  meanpressure    2   
5     humidity      humidity    1   
6   wind_speed    wind_speed    1   
7     meantemp      meantemp    1   
8     meantemp      meantemp    2   
9     meantemp      meantemp    3   
10    humidity    wind_speed    5   
11    meantemp    wind_speed    2   
12    humidity      humidity    5   
13    humidity      humidity    2   
14    humidity    wind_speed    0   
15    humidity      meantemp    0   
16    humidity    wind_speed    3   
17    meantemp      humidity    3   
18    humidity    wind_speed    4   
19    